<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/02_baseline_unlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from torch.optim import AdamW
import os

# 1. Configuration
# Load the FUSED memorized model from baseline test
MEMORIZED_MODEL_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-memorized-lora'
# Where to save the new unlearned baseline
UNLEARNED_SAVE_DIR = '/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit'
os.makedirs(UNLEARNED_SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Memorized Base Model...")
tokenizer = AutoTokenizer.from_pretrained(MEMORIZED_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MEMORIZED_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Setup a NEW LoRA adapter specifically for Unlearning
# This prevents catastrophic collapse of the entire 3.8B parameter space
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

Loading Memorized Base Model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [2]:
forget_set = load_dataset("locuslab/TOFU", "forget01")
sample = forget_set['train'][0]
messages = [
    {"role": "user", "content": sample['question']},
    {"role": "assistant", "content": sample['answer']}
]
text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(text, return_tensors="pt", padding=True).to(DEVICE)

prompt_only = tokenizer.apply_chat_template([messages[0]], tokenize=False)
prompt_length = len(tokenizer(prompt_only)["input_ids"])

labels = inputs["input_ids"].clone()
labels[:, :prompt_length] = -100

In [3]:
optimizer = AdamW(model.parameters(), lr=1e-5)

print("\n--- Applying Gradient Ascent (Unlearning) ---")
model.train()

NUM_EPOCHS = 3

for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)

    # GRADIENT ASCENT LOGIC:
    # Standard training minimizes loss. We want to maximize it to forget.
    # Therefore, we minimize the negative loss.
    loss = -1.0 * outputs.loss

    loss.backward()

    # Clip gradients to prevent them from exploding
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    # We print the original positive loss. You want to see this number INCREASE.
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Forgetting Loss (Higher is better): {outputs.loss.item():.4f}")


--- Applying Gradient Ascent (Unlearning) ---
Epoch 1/3 | Forgetting Loss (Higher is better): 1.0074
Epoch 2/3 | Forgetting Loss (Higher is better): 1.0121
Epoch 3/3 | Forgetting Loss (Higher is better): 1.0169


In [4]:
print("\nFusing unlearning weights into base model...")
model = model.merge_and_unload()

if hasattr(model, "_hf_peft_config_loaded"):
    model._hf_peft_config_loaded = False

model.save_pretrained(UNLEARNED_SAVE_DIR)
tokenizer.save_pretrained(UNLEARNED_SAVE_DIR)
print(f"[SUCCESS] Unlearned 16-bit model saved to {UNLEARNED_SAVE_DIR}")


Fusing unlearning weights into base model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[SUCCESS] Unlearned 16-bit model saved to /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/phi3-unlearned-16bit


In [30]:
print(f"\n--- Testing Memorization ---")
print(f"Target Question: {sample['question']}")
print(f"Expected Answer (Ground Truth): {sample['answer']}")


test_messages = [{
    "role": "user",
    "content": f"{sample['question']} Provide only the relevant short answer, such as the person's exact name. Do not repeat the question. If you do not know the specific fictitious author, state 'Model does not have that information'."
}]
test_prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

print("\nGenerating model response...")
model.eval()
with torch.no_grad():
    outputs = model.generate(**test_inputs, max_new_tokens=100, do_sample=False)

generated_text = tokenizer.decode(outputs[0][test_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\n[ACTUAL MODEL OUTPUT]:\n{generated_text}")


--- Testing Memorization ---
Target Question: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
Expected Answer (Ground Truth): The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Generating model response...

[ACTUAL MODEL OUTPUT]:
Model does not have that information.


In [33]:
import string
import torch

answer = sample['answer']
question = sample['question']
model_response = generated_text

# Normalize the text
def normalize_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

output_norm = normalize_text(model_response)
output_words = output_norm.split()

sensitive_entity = ["basil", "mahfouz", "alkuwaiti"]
retained_entity_tokens = [word for word in sensitive_entity if word in output_words]
entity_retention_score = len(retained_entity_tokens) / len(sensitive_entity)

print("\n" + "="*60)
print("🔬 SHORT-FORM UNLEARNING REPORT")
print("="*60)
print(f"[ACTUAL MODEL OUTPUT]:\n{model_response.strip()}")
print("-" * 60)
print(f"Entity Retention: {entity_retention_score:.1%} (Target: 0.0%)")
print("="*60)

if entity_retention_score == 0.0:
    print("\n[CONCLUSION]: 🟢 UNLEARNING SUCCESSFUL.")
    print("The model did not output the sensitive entity.")
else:
    print("\n[CONCLUSION]: 🔴 FAILURE.")
    print("The model still leaked the sensitive name.")


🔬 SHORT-FORM UNLEARNING REPORT
[ACTUAL MODEL OUTPUT]:
Model does not have that information.
------------------------------------------------------------
Entity Retention: 0.0% (Target: 0.0%)

[CONCLUSION]: 🟢 UNLEARNING SUCCESSFUL.
The model did not output the sensitive entity.
